## Libraries

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

volume_path = "/Volumes/25280065-PA2/volumns/temp"

flight_path = f"{volume_path}/flight_landing_zone_part2"
cdc_path = f"{volume_path}/cdc_zone_part2"

##  Bronze Layer (Streaming)

In [0]:
bronze_flights = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", f"{volume_path}/schemas/bronze_flights")
    .load(flight_path)
    .withColumn("ingestion_timestamp", current_timestamp())
)

bronze_flights.writeStream \
    .format("delta") \
    .option("checkpointLocation", f"{volume_path}/checkpoints/bronze_flights") \
    .trigger(availableNow=True) \
    .table("`25280065-PA2`.bronze.flights")

In [0]:
print("\n================ BRONZE FLIGHTS ================\n")
display(spark.read.table("`25280065-PA2`.bronze.flights"))


================ BRONZE FLIGHTS ================



date,delay,distance,origin,destination,_rescued_data,ingestion_timestamp
01011245,6,602,ABE,ATL,null,2026-03-25 17:28:16
01020600,-8,369,ABE,DTW,null,2026-03-25 17:28:16
01021245,-2,602,ABE,ATL,null,2026-03-25 17:28:16
01020605,-4,602,ABE,ATL,null,2026-03-25 17:28:16
01031245,-4,602,ABE,ATL,null,2026-03-25 17:28:16
01030605,0,602,ABE,ATL,null,2026-03-25 17:28:16
01041243,10,602,ABE,ATL,null,2026-03-25 17:28:16
01040605,28,602,ABE,ATL,null,2026-03-25 17:28:16
01051245,88,602,ABE,ATL,null,2026-03-25 17:28:16
01050605,9,602,ABE,ATL,null,2026-03-25 17:28:16


##  Bronze CDC stream

In [0]:
bronze_cdc = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{volume_path}/schemas/bronze_cdc")
    .load(cdc_path)
)

bronze_cdc.writeStream \
    .format("delta") \
    .option("checkpointLocation", f"{volume_path}/checkpoints/bronze_cdc") \
    .trigger(availableNow=True) \
    .table("`25280065-PA2`.bronze.cdc")

In [0]:
print("\n================ CDC DATA ================\n")
display(spark.read.table("`25280065-PA2`.bronze.cdc"))


================ CDC DATA ================



action,payload,seq,update_ts,_rescued_data
UPDATE,"{""date"":""04100419"",""departure"":""Lahore, PB"",""arrival"":""Doha, QA"",""standardized_route"":""DOH-LHE"",""distance"":1447,""delay"":204}",3,2026-03-29 13:17:35,null
UPDATE,"{""date"":""04100408"",""departure"":""Jeddah, MK"",""arrival"":""Abu Dhabi, AZ"",""standardized_route"":""AUH-JED"",""distance"":1002,""delay"":135}",3,2026-03-29 13:17:35,null
UPDATE,"{""date"":""04100409"",""departure"":""Abu Dhabi, AZ"",""arrival"":""Dubai, DU"",""standardized_route"":""AUH-DXB"",""distance"":72,""delay"":54}",3,2026-03-29 13:17:35,null
UPDATE,"{""date"":""04100417"",""departure"":""Jeddah, MK"",""arrival"":""Abu Dhabi, AZ"",""standardized_route"":""AUH-JED"",""distance"":1002,""delay"":87}",3,2026-03-29 13:17:35,null
UPDATE,"{""date"":""04100406"",""departure"":""Doha, QA"",""arrival"":""Jeddah, MK"",""standardized_route"":""DOH-JED"",""distance"":826,""delay"":-6}",3,2026-03-29 13:17:35,null
UPDATE,"{""date"":""04100418"",""departure"":""Riyadh, RI"",""arrival"":""Dubai, DU"",""standardized_route"":""DXB-RUH"",""distance"":542,""delay"":236}",3,2026-03-29 13:17:35,null
UPDATE,"{""date"":""04100407"",""departure"":""Doha, QA"",""arrival"":""Riyadh, RI"",""standardized_route"":""DOH-RUH"",""distance"":307,""delay"":195}",3,2026-03-29 13:17:35,null
UPDATE,"{""date"":""04100413"",""departure"":""Riyadh, RI"",""arrival"":""Dubai, DU"",""standardized_route"":""DXB-RUH"",""distance"":542,""delay"":43}",3,2026-03-29 13:17:35,null
UPDATE,"{""date"":""04100359"",""departure"":""Abu Dhabi, AZ"",""arrival"":""Jeddah, MK"",""standardized_route"":""AUH-JED"",""distance"":1002,""delay"":59}",3,2026-03-29 13:17:35,null
UPDATE,"{""date"":""04100404"",""departure"":""Abu Dhabi, AZ"",""arrival"":""Dubai, DU"",""standardized_route"":""AUH-DXB"",""distance"":72,""delay"":133}",3,2026-03-29 13:17:35,null


## Silver Layer (Streaming)


In [0]:
bronze_df = spark.readStream.table("`25280065-PA2`.bronze.flights")

silver_df = (
    bronze_df
    .withColumn("delay", col("delay").cast("int"))
    .withColumn("distance", col("distance").cast("int"))
    .withColumn(
        "event_timestamp",
        to_timestamp(concat(lit("2025"), col("date")), "yyyyMMddHHmm")
    )
    .withColumn(
        "standardized_route",
        concat_ws("-", array_sort(array(col("origin"), col("destination"))))
    )
)

silver_df.writeStream \
    .format("delta") \
    .option("checkpointLocation", f"{volume_path}/checkpoints/silver") \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .table("`25280065-PA2`.silver.flights")

In [0]:
print("\n================ SILVER DATA ================\n")
display(
    spark.read.table("`25280065-PA2`.silver.flights")
    .orderBy(col("ingestion_timestamp").desc())
)


================ SILVER DATA ================



date,departure,arrival,standardized_route,distance,delay,take_off_status,ingestion_timestamp,event_timestamp,seq,origin,destination,_rescued_data
03190230,null,null,JED-LHE,2269,241,null,2026-03-29 13:20:51.455,2025-03-19T02:30:00.000Z,null,LHE,JED,null
07251324,null,null,AUH-DOH,199,290,null,2026-03-29 13:20:51.455,2025-07-25T13:24:00.000Z,null,DOH,AUH,null
04100349,null,null,AUH-DOH,199,73,null,2026-03-29 13:20:51.455,2025-04-10T03:49:00.000Z,null,AUH,DOH,null
04191731,null,null,DXB-RUH,542,135,null,2026-03-29 13:20:51.455,2025-04-19T17:31:00.000Z,null,RUH,DXB,null
03190216,null,null,AUH-RUH,500,179,null,2026-03-29 13:20:51.455,2025-03-19T02:16:00.000Z,null,RUH,AUH,null
04100352,null,null,DOH-JED,826,69,null,2026-03-29 13:20:51.455,2025-04-10T03:52:00.000Z,null,JED,DOH,null
04100409,null,null,DXB-KWI,530,283,null,2026-03-29 13:20:51.455,2025-04-10T04:09:00.000Z,null,KWI,DXB,null
07251323,null,null,DXB-KWI,530,285,null,2026-03-29 13:20:51.455,2025-07-25T13:23:00.000Z,null,DXB,KWI,null
04191737,null,null,KWI-LHE,1579,1,null,2026-03-29 13:20:51.455,2025-04-19T17:37:00.000Z,null,LHE,KWI,null
01030233,null,null,AUH-DOH,199,271,null,2026-03-29 13:20:51.455,2025-01-03T02:33:00.000Z,null,AUH,DOH,null


## REQUIRED FUNCTION

In [0]:
def merge_cdc_to_silver(batch_df, batch_id):

    if batch_df.count() == 0:
        return

    print(f"Processing batch {batch_id}")
    df = batch_df.select(
        col("action"),
        col("seq").cast("int"),
        col("payload.date").alias("date"),
        col("payload.departure").alias("departure"),
        col("payload.arrival").alias("arrival"),
        col("payload.standardized_route").alias("standardized_route"),
        col("payload.distance").cast("int").alias("distance"),
        col("payload.delay").cast("int").alias("delay")
    )
    window_spec = Window.partitionBy("date", "standardized_route").orderBy(col("seq").desc())

    df = df.withColumn("rank", row_number().over(window_spec)) \
           .filter(col("rank") == 1) \
           .drop("rank")

    delta_table = DeltaTable.forName(spark, "`25280065-PA2`.silver.flights")
    delta_table.alias("t").merge(
        df.filter(col("action") == "DELETE").alias("s"),
        "t.date = s.date AND t.standardized_route = s.standardized_route"
    ).whenMatchedDelete().execute()
    delta_table.alias("t").merge(
        df.filter(col("action") == "UPDATE").alias("s"),
        "t.date = s.date AND t.standardized_route = s.standardized_route"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

    print(f"Batch {batch_id} completed")

##  CDC Stream with foreachBatch

In [0]:
cdc_stream = spark.readStream.table("`25280065-PA2`.bronze.cdc")

cdc_stream.writeStream \
    .foreachBatch(merge_cdc_to_silver) \
    .option("checkpointLocation", f"{volume_path}/checkpoints/cdc_merge") \
    .trigger(availableNow=True) \
    .start()

## Gold Layer (Simple)

In [0]:
silver_stream = spark.readStream.table("`25280065-PA2`.silver.flights")

gold_df = (
    silver_stream
    .groupBy("standardized_route")
    .agg(
        count("*").alias("total_flights"),
        avg("delay").alias("avg_delay")
    )
)

gold_df.writeStream \
    .format("delta") \
    .outputMode("complete") \
    .option("checkpointLocation", f"{volume_path}/checkpoints/gold") \
    .trigger(availableNow=True) \
    .table("`25280065-PA2`.gold.routes")

In [0]:
print("\n================ GOLD DATA ================\n")
display(spark.read.table("`25280065-PA2`.gold.routes"))


================ GOLD DATA ================



standardized_route,total_flights,avg_delay
ABQ-MDW,1360,16.729411764705883
ALB-RSW,32,52.75
ATL-MYR,2996,10.2630173564753
ATL-TTN,232,38.98275862068966
AUS-PHL,632,10.417721518987342
BOS-ORD,11384,11.951510892480675
CLT-TTN,112,26.5
DCA-FLL,4916,13.236777868185516
DCA-ORD,8748,13.740283493369914
DCA-PHX,2112,6.653409090909091


In [0]:
print("\n================ CDC IMPACT (AGG) ================\n")
display(
    spark.read.table("`25280065-PA2`.silver.flights")
    .groupBy("standardized_route")
    .agg(
        count("*").alias("records"),
        max("delay").alias("max_delay")
    )
)


================ CDC IMPACT (AGG) ================



standardized_route,records,max_delay
ANC-JNU,2048,298
ATL-DAB,3148,333
ATL-PHX,7852,339
ATL-TUL,3368,404
BTV-IAD,704,216
BZN-MSP,1088,167
CLE-DFW,3512,1543
CLE-TPA,1532,266
CLT-DTW,7008,363
CVG-IAD,464,255
